# 🎯 DMAIC Katapult-Versuch

## Qualitätsmanagement – Praxistag

Willkommen zum zweiten QM-Tag! Heute durchlauft ihr den **kompletten DMAIC-Zyklus** an eurem selbstgebauten Katapult.

**Was ihr braucht:** Katapult, Tischtennisbälle, Maßband, Klebeband, Laptop

**Ablauf:**
| Phase | Inhalt | Dauer |
|-------|--------|-------|
| Define | Zielweite, Testwürfe, Charter | 45 min |
| Measure | MSA + Baseline | 75 min |
| Analyze | DoE-Planung + Durchführung + Modell | ~4h |
| Improve | Optimierung + Konfirmation | 60 min |
| Control | Kontrollkarte, Cpk, Vorher/Nachher | 30 min |

⚠️ **Wichtig:** Zellen der Reihe nach ausführen! Keine Zellen überspringen.

In [ ]:
#@title 🔧 Bibliotheken installieren (einmalig ausführen)
!pip install -q statsmodels openpyxl
print("✅ Alle Bibliotheken installiert!")


In [ ]:
#@title 🔧 Google Drive verbinden (für Auto-Speicherung)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive verbunden – Auto-Speicherung aktiv!")
except ImportError:
    print("ℹ️ Nicht in Colab – Fortschritt wird lokal gespeichert.")


> **💾 Auto-Speicherung aktiv.** Euer Fortschritt wird nach jeder Eingabe automatisch
> in Google Drive gespeichert (`MyDrive/DMAIC_Katapult/`). Falls die Colab-Session
> abstürzt, führt die Zellen 1–4 erneut aus — eure Daten werden automatisch geladen.
>
> **Neu starten?** Gruppennamen ändern oder den Ordner in Google Drive löschen.


In [ ]:
#@title 🔧 Initialisierung
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import HTML, display
import warnings, os, importlib, sys
warnings.filterwarnings('ignore')

# helper.py von GitHub laden (immer aktuelle Version)
import urllib.request
_helper_url = "https://raw.githubusercontent.com/dozwa/HWR_QM_DOE/main/helper.py"
try:
    urllib.request.urlretrieve(_helper_url, "helper.py")
    print("✅ helper.py von GitHub geladen")
except Exception as _e:
    if os.path.exists("helper.py"):
        print(f"⚠️ GitHub nicht erreichbar – nutze lokale Version")
    else:
        print(f"❌ helper.py konnte nicht geladen werden: {_e}")
        raise

if 'helper' in sys.modules:
    importlib.reload(sys.modules['helper'])
import helper
helper.setup_theme()
print("✅ Initialisierung abgeschlossen!")


In [ ]:
#@title 📝 Projekt einrichten
modus = "Neues Projekt" #@param ["Neues Projekt", "Fortschritt laden"]
gruppenname = "Gruppe A" #@param {type:"string"}
gruppennummer = 1 #@param {type:"integer"}
zielweite = 300.0 #@param {type:"number"}
toleranz = 15 #@param {type:"number"}
speicherstand_nr = 0 #@param {type:"integer"}

if modus == "Fortschritt laden":
    _staende = helper.finde_speicherstaende()
    if not _staende:
        print("❌ Keine Speicherstände im Google Drive gefunden.")
        print("   Wechsle zu 'Neues Projekt' um ein neues Projekt zu starten.")
    elif speicherstand_nr == 0:
        helper.zeige_speicherstand_auswahl(_staende)
        print("\n👆 Trage die gewünschte Nr. bei 'speicherstand_nr' ein und führe diese Zelle erneut aus.")
    else:
        if speicherstand_nr > len(_staende):
            print(f"❌ Nr. {speicherstand_nr} existiert nicht (max. {len(_staende)}).")
        else:
            _s = _staende[speicherstand_nr - 1]
            projekt = helper.lade_fortschritt(_s["gruppenname"], _s["gruppennummer"])
            if projekt is not None:
                helper.zeige_restore_zusammenfassung(projekt)
            else:
                print("❌ Laden fehlgeschlagen.")
else:
    # Neues Projekt oder auto-detect existierenden Speicherstand
    _restored = helper.lade_fortschritt(gruppenname, gruppennummer)
    if _restored is not None:
        projekt = _restored
        helper.zeige_restore_zusammenfassung(projekt)
    else:
        projekt = helper.init_projekt(gruppenname, gruppennummer,
                                      zielweite=zielweite, toleranz=toleranz)
        helper.speichere_fortschritt(projekt)
        print(f"✅ Neues Projekt initialisiert!")
        print(f"   Gruppe: {projekt.gruppenname} (Nr. {projekt.gruppennummer})")
        print(f"   Zielweite: {projekt.zielweite:.0f} cm ± {projekt.toleranz:.0f} cm")
        print(f"   (Zielweite könnt ihr nach der Katapult-Vermessung unten noch anpassen.)")


---
# Phase 1: DEFINE

## Was ist das Problem? Wie schlecht ist es?

In der Define-Phase beschreibt ihr zunächst euer Katapult, vermesst dessen Reichweite und legt dann die **Zielweite** fest, die ihr reproduzierbar treffen wollt.

## Schritt 1 – Faktoren definieren

Welche **Einstellungen** lassen sich an eurem Katapult verändern? Tragt Name, Einheit und die beiden geplanten Stufen (Low/High) ein. Diese Faktoren werden gleich für die Min/Max-Vermessung und später in ANALYZE für den Versuchsplan verwendet.

> **Stetig vs. zweistufig:** Wenn der Faktor frei einstellbar ist (z.B. Winkel, Spannung), `centerpoint` auf `True` lassen. Bei nur zwei festen Positionen auf `False`.

In [ ]:
#@title 📝 Faktor 1 definieren
f1_name = "Winkel" #@param {type:"string"}
f1_einheit = "Grad" #@param {type:"string"}
f1_low = 30.0 #@param {type:"number"}
f1_high = 45.0 #@param {type:"number"}
f1_centerpoint = True #@param {type:"boolean"}

faktor1 = {"name": f1_name, "einheit": f1_einheit, "low": f1_low, "high": f1_high, "centerpoint_moeglich": f1_centerpoint}
_cp = "stetig → Centerpoint möglich" if f1_centerpoint else "zweistufig → kein Centerpoint"
print(f"✅ Faktor 1: {f1_name} [{f1_low} – {f1_high} {f1_einheit}] ({_cp})")


In [ ]:
#@title 📝 Faktor 2 definieren
f2_name = "Spannung" #@param {type:"string"}
f2_einheit = "cm" #@param {type:"string"}
f2_low = 5.0 #@param {type:"number"}
f2_high = 15.0 #@param {type:"number"}
f2_centerpoint = True #@param {type:"boolean"}

faktor2 = {"name": f2_name, "einheit": f2_einheit, "low": f2_low, "high": f2_high, "centerpoint_moeglich": f2_centerpoint}
_cp = "stetig → Centerpoint möglich" if f2_centerpoint else "zweistufig → kein Centerpoint"
print(f"✅ Faktor 2: {f2_name} [{f2_low} – {f2_high} {f2_einheit}] ({_cp})")


In [ ]:
#@title 📝 Faktor 3 definieren
f3_name = "Ballposition" #@param {type:"string"}
f3_einheit = "cm" #@param {type:"string"}
f3_low = 0.0 #@param {type:"number"}
f3_high = 5.0 #@param {type:"number"}
f3_centerpoint = True #@param {type:"boolean"}

faktor3 = {"name": f3_name, "einheit": f3_einheit, "low": f3_low, "high": f3_high, "centerpoint_moeglich": f3_centerpoint}
_cp = "stetig → Centerpoint möglich" if f3_centerpoint else "zweistufig → kein Centerpoint"
print(f"✅ Faktor 3: {f3_name} [{f3_low} – {f3_high} {f3_einheit}] ({_cp})")


In [ ]:
#@title 📝 Faktor 4 (optional – leer lassen wenn nur 3 Faktoren)
f4_name = "" #@param {type:"string"}
f4_einheit = "" #@param {type:"string"}
f4_low = 0.0 #@param {type:"number"}
f4_high = 0.0 #@param {type:"number"}
f4_centerpoint = True #@param {type:"boolean"}

if f4_name.strip():
    faktor4 = {"name": f4_name, "einheit": f4_einheit, "low": f4_low, "high": f4_high, "centerpoint_moeglich": f4_centerpoint}
    _cp = "stetig → Centerpoint möglich" if f4_centerpoint else "zweistufig → kein Centerpoint"
    print(f"✅ Faktor 4: {f4_name} [{f4_low} – {f4_high} {f4_einheit}] ({_cp})")
else:
    faktor4 = None
    print("ℹ️ Kein 4. Faktor definiert (3-Faktor-Design)")


In [ ]:
#@title 📝 Faktor 5 (optional – leer lassen wenn nur 3–4 Faktoren)
f5_name = "" #@param {type:"string"}
f5_einheit = "" #@param {type:"string"}
f5_low = 0.0 #@param {type:"number"}
f5_high = 0.0 #@param {type:"number"}
f5_centerpoint = True #@param {type:"boolean"}

if f5_name.strip():
    faktor5 = {"name": f5_name, "einheit": f5_einheit, "low": f5_low, "high": f5_high, "centerpoint_moeglich": f5_centerpoint}
    _cp = "stetig → Centerpoint möglich" if f5_centerpoint else "zweistufig → kein Centerpoint"
    print(f"✅ Faktor 5: {f5_name} [{f5_low} – {f5_high} {f5_einheit}] ({_cp})")
else:
    faktor5 = None
    print("ℹ️ Kein 5. Faktor definiert")


In [ ]:
#@title 📋 Faktoren übernehmen
_faktoren_neu = [faktor1, faktor2, faktor3]
try:
    if faktor4 and faktor4.get("name"):
        _faktoren_neu.append(faktor4)
except NameError:
    pass
try:
    if faktor5 and faktor5.get("name"):
        _faktoren_neu.append(faktor5)
except NameError:
    pass

for f in _faktoren_neu:
    if f["low"] == f["high"]:
        print(f"❌ Faktor '{f['name']}': Low und High sind identisch ({f['low']}). Bitte unterschiedliche Stufen wählen!")
    elif f["low"] > f["high"]:
        f["low"], f["high"] = f["high"], f["low"]
        print(f"⚠️ Faktor '{f['name']}': Low und High vertauscht – automatisch korrigiert.")

projekt.faktoren = _faktoren_neu
print(f"✅ {len(projekt.faktoren)} Faktoren übernommen:")
for f in projekt.faktoren:
    print(f"   • {f['name']} [{f['low']} – {f['high']} {f['einheit']}]")
helper.speichere_fortschritt(projekt)


## Schritt 2 – Katapult vermessen

Ermittelt die **Reichweiten-Spanne** eures Katapults:

1. Stellt alle Faktoren so ein, dass ihr die **kürzeste** Wurfweite erwartet. Werft 3× und tragt die Werte unten ein.
2. Stellt alle Faktoren so ein, dass ihr die **längste** Wurfweite erwartet. Werft 3× und tragt ein.
3. Die tatsächlich eingestellten Werte pro Faktor werden ebenfalls dokumentiert – vorbelegt sind Low (Min) bzw. High (Max) eurer Faktordefinition.

> So erhaltet ihr einen realistischen Bereich, in dem ihr gleich die Zielweite festlegt, und eine reproduzierbare Katapult-Beschreibung.

In [ ]:
#@title 🎯 Min-Konfiguration eintragen
# Einstellungen bei Min-Konfiguration (pro Faktor)
# Trage hier die tatsächlich eingestellten Werte ein. Wenn du 0 stehen lässt,
# wird der Low-Wert aus der Faktor-Definition übernommen.
min_val_1 = 0.0 #@param {type:"number"}
min_val_2 = 0.0 #@param {type:"number"}
min_val_3 = 0.0 #@param {type:"number"}
min_val_4 = 0.0 #@param {type:"number"}
min_val_5 = 0.0 #@param {type:"number"}

# Drei Würfe bei Min-Konfiguration
min_wurf_1 = 0.0 #@param {type:"number"}
min_wurf_2 = 0.0 #@param {type:"number"}
min_wurf_3 = 0.0 #@param {type:"number"}

_min_vals_raw = [min_val_1, min_val_2, min_val_3, min_val_4, min_val_5]
_min_einst = {}
for i, f in enumerate(projekt.faktoren):
    _min_einst[f["name"]] = _min_vals_raw[i] if _min_vals_raw[i] != 0.0 else f["low"]
_min_wuerfe = [min_wurf_1, min_wurf_2, min_wurf_3]

if any(w > 0 for w in _min_wuerfe):
    helper.speichere_vermessung(
        projekt,
        min_wuerfe=_min_wuerfe,
        max_wuerfe=[],
        min_einstellung=_min_einst,
        max_einstellung={},
    )
    print(f"✅ Min-Konfiguration gespeichert (μ={float(np.mean(projekt.vermessung_min_wuerfe)):.1f} cm).")
elif len(projekt.vermessung_min_wuerfe) > 0:
    print(f"ℹ️ Min-Würfe aus gespeichertem Fortschritt: μ={float(np.mean(projekt.vermessung_min_wuerfe)):.1f} cm.")
else:
    print("⚠️ Bitte mindestens einen Min-Wurf eingeben (Wert > 0).")


In [ ]:
#@title 🎯 Max-Konfiguration eintragen
# Einstellungen bei Max-Konfiguration (pro Faktor)
# Trage hier die tatsächlich eingestellten Werte ein. Wenn du 0 stehen lässt,
# wird der High-Wert aus der Faktor-Definition übernommen.
max_val_1 = 0.0 #@param {type:"number"}
max_val_2 = 0.0 #@param {type:"number"}
max_val_3 = 0.0 #@param {type:"number"}
max_val_4 = 0.0 #@param {type:"number"}
max_val_5 = 0.0 #@param {type:"number"}

# Drei Würfe bei Max-Konfiguration
max_wurf_1 = 0.0 #@param {type:"number"}
max_wurf_2 = 0.0 #@param {type:"number"}
max_wurf_3 = 0.0 #@param {type:"number"}

beschreibung = "" #@param {type:"string"}

_max_vals_raw = [max_val_1, max_val_2, max_val_3, max_val_4, max_val_5]
_max_einst = {}
for i, f in enumerate(projekt.faktoren):
    _max_einst[f["name"]] = _max_vals_raw[i] if _max_vals_raw[i] != 0.0 else f["high"]
_max_wuerfe = [max_wurf_1, max_wurf_2, max_wurf_3]

if any(w > 0 for w in _max_wuerfe):
    helper.speichere_vermessung(
        projekt,
        min_wuerfe=[],
        max_wuerfe=_max_wuerfe,
        min_einstellung={},
        max_einstellung=_max_einst,
        beschreibung=beschreibung,
    )
    print(f"✅ Max-Konfiguration gespeichert (μ={float(np.mean(projekt.vermessung_max_wuerfe)):.1f} cm).")
elif len(projekt.vermessung_max_wuerfe) > 0:
    print(f"ℹ️ Max-Würfe aus gespeichertem Fortschritt: μ={float(np.mean(projekt.vermessung_max_wuerfe)):.1f} cm.")
else:
    print("⚠️ Bitte mindestens einen Max-Wurf eingeben (Wert > 0).")


In [ ]:
#@title 📊 Vermessung anzeigen
helper.zeige_vermessung(projekt)

## Schritt 3 – Zielweite festlegen

Eure Zielweite ist aktuell **300 cm** (Default aus der Projekt-Einrichtung). Passt den Wert an, wenn euer Katapult die 300 cm nicht stabil erreicht oder ihr eine andere Herausforderung wählen wollt. Die Toleranz (siehe oben) definiert zusammen mit der Zielweite die **Testgrenzen** (LSL/USL), auf die wir in der CONTROL-Phase (Cpk) zurückkommen.

In [ ]:
#@title 🎯 Zielweite anpassen (optional)
neue_zielweite = 300.0 #@param {type:"number"}
helper.setze_zielweite(projekt, neue_zielweite)


In [ ]:
#@title 🎯 Eure Zielweite
print(f"{'='*50}")
print(f"  EURE AUFGABE")
print(f"  Zielweite:  {projekt.zielweite:.0f} cm")
print(f"  Toleranz:   ±{projekt.toleranz:.0f} cm")
print(f"  Zielband:   [{projekt.zielweite - projekt.toleranz:.0f}, {projekt.zielweite + projekt.toleranz:.0f}] cm")
print(f"{'='*50}")
print(f"\nFindet per OFAT-Annäherung eine Einstellung nahe der Zielweite")
print(f"und macht dann 5 Testwürfe, um die Streuung (CV) zu bewerten.")

## Schritt 4 – Initiale Einstellung finden (OFAT)

Bevor ihr Testwürfe macht, **sucht** ihr am Katapult eine Einstellung, mit der ihr der Zielweite schon nahe kommt. Geht dabei nach **One-Factor-At-A-Time (OFAT)** vor:

1. Startet bei einer mittleren Einstellung (z.B. jeweils zwischen Low und High eurer Faktor-Definition)
2. Macht einen Probewurf und vergleicht mit der Zielweite
3. **Ändert pro Iteration nur einen Faktor** und werft erneut
4. Wiederholt, bis ihr mit einem Wurf nahe an der Zielweite liegt
5. Tragt die gefundene Einstellung unten ein – sie gilt für die 5 Testwürfe und später die Baseline

> 💡 **Warum nur ein Faktor pro Iteration?** Dann könnt ihr eindeutig sehen, wie sich *dieser* Faktor auswirkt. Wechselwirkungen zwischen Faktoren findet später das DoE in ANALYZE — hier geht es um ein erstes Gefühl.

In [ ]:
#@title 📝 Initiale Einstellung eintragen
# Tragt die durch OFAT gefundene Einstellung ein.
# Felder für nicht genutzte Faktoren bleiben auf 0.
init_val_1 = 0.0 #@param {type:"number"}
init_val_2 = 0.0 #@param {type:"number"}
init_val_3 = 0.0 #@param {type:"number"}
init_val_4 = 0.0 #@param {type:"number"}
init_val_5 = 0.0 #@param {type:"number"}

_init_vals_raw = [init_val_1, init_val_2, init_val_3, init_val_4, init_val_5]
_init_einst = {f["name"]: _init_vals_raw[i] for i, f in enumerate(projekt.faktoren)}

if any(v != 0.0 for v in _init_einst.values()):
    helper.setze_initiale_einstellung(projekt, _init_einst)
elif projekt.initiale_einstellung:
    print("ℹ️ Initiale Einstellung aus gespeichertem Fortschritt:")
    for name, val in projekt.initiale_einstellung.items():
        einheit = next((f["einheit"] for f in projekt.faktoren if f["name"] == name), "")
        print(f"   • {name}: {val} {einheit}")
else:
    print("⚠️ Bitte die gefundenen Einstellungswerte eintragen.")


## Schritt 5 – Testwürfe (CV bei initialer Einstellung)

1. Behaltet die eben dokumentierte **initiale Einstellung** bei
2. Macht **5 Würfe** und messt die Wurfweite
3. Tragt die Werte unten ein

> ⚠️ Noch keine Optimierung – es geht hier um die Reproduzierbarkeit eurer Ausgangseinstellung (CV).

In [ ]:
#@title 📝 Testwürfe eingeben
wurf_1 = 0.0 #@param {type:"number"}
wurf_2 = 0.0 #@param {type:"number"}
wurf_3 = 0.0 #@param {type:"number"}
wurf_4 = 0.0 #@param {type:"number"}
wurf_5 = 0.0 #@param {type:"number"}

wuerfe = np.array([wurf_1, wurf_2, wurf_3, wurf_4, wurf_5])
wuerfe = wuerfe[wuerfe > 0]  # Nullen entfernen

if len(wuerfe) > 0:
    projekt.testwuerfe = wuerfe
    print(f"✅ {len(wuerfe)} Testwürfe eingetragen: {wuerfe}")
    helper.speichere_fortschritt(projekt)
elif len(projekt.testwuerfe) > 0:
    print(f"ℹ️ {len(projekt.testwuerfe)} Testwürfe aus gespeichertem Fortschritt geladen.")
else:
    print("⚠️ Bitte mindestens einen Testwurf eingeben!")


In [ ]:
#@title 📊 Testwurf-Auswertung
helper.zeige_testwurf_ergebnis(projekt)

### Projektcharter

Die Projektcharter dokumentiert euer Qualitätsverbesserungsprojekt. Sie ist das zentrale Dokument der Define-Phase.

> 📋 **Für den Bericht:** Die Projektcharter gehört vollständig in euren Bericht!

In [ ]:
#@title 📝 Projektcharter ausfüllen
problemstellung = "z.B. Katapult trifft die Zielweite nicht reproduzierbar" #@param {type:"string"}
projektziel = "z.B. Wurfweite 450 cm ± 15 cm mit Cpk > 1.0" #@param {type:"string"}
scope = "z.B. Optimierung der Faktoreinstellungen, nicht der Konstruktion" #@param {type:"string"}
projektleiter = "" #@param {type:"string"}
protokollant = "" #@param {type:"string"}
versuchsdurchfuehrende = "" #@param {type:"string"}

projekt.charter = {
    "Gruppenname": projekt.gruppenname,
    "Zielweite": f"{projekt.zielweite:.0f} cm ± {projekt.toleranz:.0f} cm",
    "Problemstellung": problemstellung,
    "Projektziel": projektziel,
    "Scope": scope,
    "Projektleiter/in": projektleiter,
    "Protokollant/in": protokollant,
    "Versuchsdurchführende": versuchsdurchfuehrende,
    "Datum": "23.04.2026",
}

display(HTML(helper.formatiere_charter(projekt)))
helper.speichere_fortschritt(projekt)


<div style="padding:10px; border-left:4px solid #2563EB; background:#DBEAFE; border-radius:4px;">
📋 <strong>Für den Bericht:</strong> Die Projektcharter und die Zielscheibe der Testwürfe gehören in die Define-Phase eures Berichts. Exportiert die Grafiken am Ende des Tages.
</div>

<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Was ist der Variationskoeffizient (CV)?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

Der **Variationskoeffizient** (CV) setzt die Standardabweichung ins Verhältnis zum Mittelwert:

$$CV = \frac{\sigma}{\bar{x}} \times 100\%$$

Er ist ein **dimensionsloser** Streuungsmaß – damit kann man die Streuung verschiedener Prozesse vergleichen, unabhängig von der absoluten Größe.

**Interpretation für Katapulte:**
- CV < 15%: Katapult ist reproduzierbar
- CV 15-30%: Grenzwertig
- CV > 30%: Grundlegendes Problem mit der Reproduzierbarkeit
</div>
</details>

In [ ]:
#@title 💾 Ergebnisse bis DEFINE auf Google Drive speichern
helper.exportiere_phase_auf_drive(projekt, "DEFINE")


---
# Phase 2: MEASURE

## Können wir unserer Messung vertrauen?

Bevor ihr optimiert, müsst ihr sicherstellen, dass euer **Messsystem zuverlässig** ist. Sonst könnten Unterschiede in euren Daten vom Messen kommen – nicht vom Katapult.

**Zwei Übungen:**
1. **Type-1 Studie:** Wie gut ist euer Maßband? (Wiederholbarkeit)
2. **Reproduzierbarkeitsstudie:** Messen alle gleich?

### Messsystemanalyse (MSA) – Erinnerung

Die MSA prüft zwei Dinge:
- **Wiederholbarkeit (Repeatability):** Selbe Person, selbe Messung → gleiches Ergebnis?
- **Reproduzierbarkeit (Reproducibility):** Verschiedene Personen → gleiches Ergebnis?

> Euer Messergebnis ist nicht die Wahrheit, sondern eine **Schätzung**. Die MSA prüft, wie gut diese Schätzung ist.

**Varianzzerlegung:**
$$\sigma^2_{total} = \sigma^2_{Teil} + \sigma^2_{Mess}$$
$$\sigma^2_{Mess} = \sigma^2_{Wiederhol} + \sigma^2_{Reprod}$$

In [ ]:
#@title 📊 Referenz: Accuracy vs. Precision (4 Quadranten)
fig = helper.plot_4_zielscheiben_referenz()
helper._save_fig(projekt, fig, "measure_zielscheiben_referenz")
plt.show()

### MSA-Durchführung

#### Übung A: Type-1 (Messmittelfähigkeit)
1. Markiert einen **festen Punkt** auf dem Boden (Klebeband bei z.B. 400 cm)
2. **Jede Person** misst diesen Punkt **10× unabhängig** mit dem Maßband
3. **Verdeckt notieren** – ohne die Werte der anderen zu sehen!

#### Übung B: Reproduzierbarkeit
1. Macht **10 Würfe** mit identischen Katapult-Einstellungen
2. Nach jedem Wurf: **Auftreffpunkt sofort markieren** (Kreppband)
3. **Jede Person** misst unabhängig die Distanz zum Auftreffpunkt
4. **Keine Kommunikation** während der Messung!
5. Erst alle Messungen notiert → dann Markierung entfernen → nächster Wurf

In [ ]:
#@title 📥 MSA-Template herunterladen
anzahl_personen = 3 #@param {type:"integer"}

filepath = helper.generate_msa_template(
    gruppenname=projekt.gruppenname,
    num_personen=anzahl_personen,
    messmodus=projekt.messmodus,
)
print(f"✅ Template erstellt: {filepath}")

# Kopie im Drive-Ordner ablegen
import shutil
_save_dir = helper._fortschritt_verzeichnis(projekt)
if _save_dir:
    os.makedirs(_save_dir, exist_ok=True)
    shutil.copy2(filepath, os.path.join(_save_dir, os.path.basename(filepath)))

try:
    from google.colab import files
    files.download(filepath)
    print("📥 Download gestartet!")
except ImportError:
    print(f"📁 Datei gespeichert unter: {filepath}")


### MSA-Daten hochladen

Füllt das Excel-Template aus und ladet es hier hoch. Das Notebook wertet eure MSA automatisch aus.

In [ ]:
#@title ⬆️ MSA-Daten hochladen und Type-1 auswerten
try:
    from google.colab import files
    print("⬆️ Bitte MSA_Messung_Template.xlsx (ausgefüllt) hochladen:")
    uploaded = files.upload()
    msa_file = list(uploaded.keys())[0]
except ImportError:
    msa_file = "MSA_Messung_Template.xlsx"

# Type-1 auslesen
import openpyxl
wb = openpyxl.load_workbook(msa_file, data_only=True)
ws_type1 = wb["Type-1"]

# Header in Zeile 8, Daten ab Zeile 9
type1_data = {}
for row in ws_type1.iter_rows(min_row=9, max_row=18, values_only=True):
    if row[0] is not None:
        nr = row[0]
        ref = row[1] if isinstance(row[1], (int, float)) else 400
        for i, val in enumerate(row[2:], start=1):
            if val is not None and isinstance(val, (int, float)):
                if f"Person {i}" not in type1_data:
                    type1_data[f"Person {i}"] = []
                type1_data[f"Person {i}"].append(float(val))

if type1_data:
    type1_df = pd.DataFrame(type1_data)
    referenzwert = ref if isinstance(ref, (int, float)) else 400.0
    type1_ergebnis = helper.analysiere_type1(type1_df, referenzwert)
    projekt.msa_type1 = type1_ergebnis
    helper.zeige_type1(type1_ergebnis)
    print(f"\n✅ Type-1 Analyse abgeschlossen (Referenzwert: {referenzwert} cm)")
else:
    print("⚠️ Keine Type-1 Daten gefunden. Bitte Excel prüfen.")

# Excel-Backup auf Drive speichern
import shutil
_save_dir = helper._fortschritt_verzeichnis(projekt)
if _save_dir:
    os.makedirs(_save_dir, exist_ok=True)
    shutil.copy2(msa_file, os.path.join(_save_dir, "MSA_Messung_Template.xlsx"))

helper.speichere_fortschritt(projekt)


In [ ]:
#@title 📊 Gage R&R Analyse (ANOVA-Methode, AIAG-konform)
try:
    ws_repr = wb["Reproduzierbarkeit"]
    
    # Header in Zeile 8, Daten ab Zeile 9
    grr_records = []
    for row in ws_repr.iter_rows(min_row=9, max_row=18, values_only=True):
        if row[0] is not None and isinstance(row[0], (int, float)):
            wurf_id = row[0]
            for i, val in enumerate(row[1:], start=1):
                if val is not None and isinstance(val, (int, float)):
                    grr_records.append({
                        "Wurf_ID": int(wurf_id),
                        "Person": f"Person {i}",
                        "Messwert": float(val),
                    })
    
    if grr_records:
        grr_df = pd.DataFrame(grr_records)
        projekt.msa_rohdaten = grr_df
        grr_ergebnis = helper.analysiere_gage_rr(grr_df)
        projekt.msa_grr = grr_ergebnis
        helper.zeige_gage_rr(grr_ergebnis)
        helper.speichere_fortschritt(projekt)
    else:
        print("⚠️ Keine Reproduzierbarkeitsdaten gefunden.")
except Exception as e:
    print(f"❌ Fehler bei der Gage R&R Analyse: {e}")
    print("Prüft eure MSA-Excel: Sind Daten im Blatt 'Reproduzierbarkeit' vorhanden?")


In [ ]:
#@title 📊 MSA-Visualisierungen
if projekt.msa_rohdaten is not None:
    grr_df = projekt.msa_rohdaten

    # Boxplot
    fig1 = helper.plot_msa_boxplot(grr_df)
    helper._save_fig(projekt, fig1, "measure_msa_boxplot")
    plt.show()

    # Interaktionsplot
    fig2 = helper.plot_msa_interaktion(grr_df)
    helper._save_fig(projekt, fig2, "measure_msa_interaktion")
    plt.show()

    # Zielscheiben pro Messer
    fig3 = helper.plot_msa_zielscheiben_pro_messer(grr_df)
    helper._save_fig(projekt, fig3, "measure_msa_zielscheiben_pro_messer")
    plt.show()
else:
    print("ℹ️ Keine MSA-Rohdaten vorhanden. Bitte zuerst Gage R&R durchführen.")


### Prozess-Baseline

Jetzt erhebt ihr euren **Ist-Zustand**: Wie gut trifft euer Katapult aktuell?

1. Stellt euer Katapult auf die **initiale Einstellung** (aus DEFINE)
2. Macht **mindestens 10, gern bis zu 20 Würfe** (mehr Würfe = aussagekräftigere Baseline)
3. Messt jede Wurfweite und tragt sie unten ein

> Die Baseline ist euer Referenzpunkt. Am Ende des Tages vergleichen wir: **Baseline vs. Konfirmation**.

In [ ]:
#@title 📝 Baseline-Würfe eingeben (Weite in cm, 1 Nachkommastelle)
wurf_01 = 0.0 #@param {type:"number"}
wurf_02 = 0.0 #@param {type:"number"}
wurf_03 = 0.0 #@param {type:"number"}
wurf_04 = 0.0 #@param {type:"number"}
wurf_05 = 0.0 #@param {type:"number"}
wurf_06 = 0.0 #@param {type:"number"}
wurf_07 = 0.0 #@param {type:"number"}
wurf_08 = 0.0 #@param {type:"number"}
wurf_09 = 0.0 #@param {type:"number"}
wurf_10 = 0.0 #@param {type:"number"}
wurf_11 = 0.0 #@param {type:"number"}
wurf_12 = 0.0 #@param {type:"number"}
wurf_13 = 0.0 #@param {type:"number"}
wurf_14 = 0.0 #@param {type:"number"}
wurf_15 = 0.0 #@param {type:"number"}
wurf_16 = 0.0 #@param {type:"number"}
wurf_17 = 0.0 #@param {type:"number"}
wurf_18 = 0.0 #@param {type:"number"}
wurf_19 = 0.0 #@param {type:"number"}
wurf_20 = 0.0 #@param {type:"number"}

_alle = [wurf_01, wurf_02, wurf_03, wurf_04, wurf_05,
         wurf_06, wurf_07, wurf_08, wurf_09, wurf_10,
         wurf_11, wurf_12, wurf_13, wurf_14, wurf_15,
         wurf_16, wurf_17, wurf_18, wurf_19, wurf_20]
werte = [round(w, 1) for w in _alle if w > 0]

if werte:
    projekt.baseline_wuerfe = np.array(werte)
    print(f"✅ {len(werte)} Baseline-Würfe eingetragen")
    print(f"   Werte: {werte}")
    helper.speichere_fortschritt(projekt)
elif len(projekt.baseline_wuerfe) > 0:
    print(f"ℹ️ {len(projekt.baseline_wuerfe)} Baseline-Würfe aus gespeichertem Fortschritt geladen.")
else:
    print("⚠️ Bitte mindestens 10 Baseline-Würfe eingeben (Weite in cm mit 1 Nachkommastelle, z.B. 345.2).")


In [ ]:
#@title 📊 Baseline-Auswertung
if len(projekt.baseline_wuerfe) > 0:
    projekt.baseline_stats = helper.analysiere_baseline(projekt.baseline_wuerfe)
    b = projekt.baseline_stats

    # Histogramm
    fig = helper.plot_baseline_histogramm(
        projekt.baseline_wuerfe, projekt.zielweite, projekt.toleranz
    )
    helper._save_fig(projekt, fig, "measure_baseline_histogramm")
    plt.show()

    # Shapiro-Wilk Bewertung
    if not np.isnan(b['shapiro_p']):
        shapiro_schwellen = [
            (0.05, "⚠️", "Abweichung von Normalverteilung – Daten und Histogramm prüfen"),
            (float('inf'), "✅", "Normalverteilung kann angenommen werden"),
        ]
        helper.zeige_ampel(b['shapiro_p'], shapiro_schwellen,
                          titel="Shapiro-Wilk p-Wert:")

    # Zielscheibe
    fig2 = helper.plot_zielscheibe(
        projekt.baseline_wuerfe, projekt.zielweite, projekt.toleranz,
        modus=projekt.messmodus, titel="Baseline – Ist-Zustand"
    )
    helper._save_fig(projekt, fig2, "measure_baseline_zielscheibe")
    plt.show()

    helper.hinweis_bericht("Baseline-Histogramm und Zielscheibe sind die Ausgangsbasis für die Bewertung des Verbesserungserfolgs.")
else:
    print("⚠️ Bitte zuerst Baseline-Würfe eingeben!")

<div style="padding:10px; border-left:4px solid #2563EB; background:#DBEAFE; border-radius:4px;">
📋 <strong>Für den Bericht:</strong> MSA-Ergebnisse (Type-1, Gage R&R) und Baseline-Histogramm sind die zentralen Outputs der Measure-Phase.
</div>

<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Warum brauchen wir eine Messsystemanalyse?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

Stellt euch vor, euer Katapult wirft tatsächlich immer auf 450 cm – aber euer Messsystem schwankt um ±20 cm. Dann sieht es so aus, als wäre das Katapult schlecht, obwohl nur die Messung schlecht ist.

Die MSA trennt die **Messunsicherheit** von der **echten Prozessvariation**. Nur so könnt ihr sicher sein, dass Verbesserungen am Katapult auch wirklich Verbesserungen sind – und nicht nur Messartefakte.

**Faustregel (AIAG):**
- %GRR < 10%: Messsystem geeignet
- %GRR 10–30%: Bedingt geeignet
- %GRR > 30%: Nicht geeignet – zuerst Messung verbessern!
</div>
</details>

In [ ]:
#@title 💾 Ergebnisse bis MEASURE auf Google Drive speichern
helper.exportiere_phase_auf_drive(projekt, "MEASURE")


---
# Phase 3: ANALYZE – Planung

## Welche Faktoren beeinflussen die Wurfweite?

Die Faktoren habt ihr in DEFINE bereits festgelegt. Hier **verfeinert** ihr sie für das DoE: Welche Faktoren sollen in den Versuchsplan, und bei welchen sind Centerpoints (mittlere Stufe) sinnvoll?

**Schritt 1:** Faktoren für das DoE verfeinern (Teilmenge + Centerpoint-Entscheidung)
**Schritt 2:** Versuchsplan generieren
**Schritt 3:** Versuche durchführen
**Schritt 4:** Modell auswerten

In [ ]:
#@title 🧩 Faktoren für das DoE verfeinern
# Aktive Faktoren und Centerpoint-Entscheidung (vorbelegt aus DEFINE)
faktor1_aktiv = True if len(projekt.faktoren) > 0 else False #@param {type:"boolean"}
faktor1_centerpoint = True #@param {type:"boolean"}
faktor2_aktiv = True if len(projekt.faktoren) > 1 else False #@param {type:"boolean"}
faktor2_centerpoint = True #@param {type:"boolean"}
faktor3_aktiv = True if len(projekt.faktoren) > 2 else False #@param {type:"boolean"}
faktor3_centerpoint = True #@param {type:"boolean"}
faktor4_aktiv = True if len(projekt.faktoren) > 3 else False #@param {type:"boolean"}
faktor4_centerpoint = True #@param {type:"boolean"}
faktor5_aktiv = True if len(projekt.faktoren) > 4 else False #@param {type:"boolean"}
faktor5_centerpoint = True #@param {type:"boolean"}

if not projekt.faktoren:
    print("❌ In DEFINE wurden noch keine Faktoren definiert. Bitte zurück zu DEFINE → 'Faktoren übernehmen'.")
else:
    _aktiv_flags = [faktor1_aktiv, faktor2_aktiv, faktor3_aktiv, faktor4_aktiv, faktor5_aktiv]
    _cp_flags = [faktor1_centerpoint, faktor2_centerpoint, faktor3_centerpoint,
                 faktor4_centerpoint, faktor5_centerpoint]

    _faktoren_doe = []
    for i, f in enumerate(projekt.faktoren):
        if not _aktiv_flags[i]:
            continue
        f_copy = dict(f)
        f_copy["centerpoint_moeglich"] = bool(_cp_flags[i] and f.get("centerpoint_moeglich", True))
        _faktoren_doe.append(f_copy)

    if len(_faktoren_doe) < 3:
        print(f"❌ Mindestens 3 aktive Faktoren nötig (derzeit {len(_faktoren_doe)}).")
    else:
        projekt.faktoren_doe = _faktoren_doe
        print(f"✅ {len(_faktoren_doe)} Faktoren für das DoE gewählt:")
        for f in _faktoren_doe:
            cp = "mit CP" if f["centerpoint_moeglich"] else "ohne CP"
            print(f"   • {f['name']} [{f['low']} – {f['high']} {f['einheit']}] ({cp})")
        helper.speichere_fortschritt(projekt)


In [ ]:
#@title ⚙️ Versuchsplan generieren
design_typ = "Vollfaktoriell (2^k)" #@param ["Vollfaktoriell (2^k)", "Halbfraktionell (2^(k-1))", "Viertelfraktionell (2^(k-2))"]
wiederholungen = 3 #@param {type:"integer"}
blocking = False #@param {type:"boolean"}
centerpoints = 3 #@param {type:"integer"}

# Design-Typ mappen
_design_map = {"Vollfaktoriell": "voll", "Halbfraktionell": "halb", "Viertelfraktionell": "viertel"}
_design = next(v for k, v in _design_map.items() if k in design_typ)

# Wiederholungen validieren
wiederholungen = max(1, min(10, int(wiederholungen)))

# DoE-Faktoren aus ANALYZE-Verfeinerung (Fallback: Master-Liste aus DEFINE)
_fak_doe = helper._effektive_faktoren(projekt)
if not _fak_doe:
    print("❌ Keine Faktoren verfügbar. Bitte zuerst DEFINE → 'Faktoren übernehmen' ausführen.")
    raise RuntimeError("keine Faktoren")

# Validierung
for f in _fak_doe:
    if f["low"] == f["high"]:
        print(f"❌ Faktor '{f['name']}': Low und High sind identisch ({f['low']}). Bitte Faktor-Definition in DEFINE anpassen!")
    if f["low"] > f["high"]:
        f["low"], f["high"] = f["high"], f["low"]
        print(f"⚠️ Faktor '{f['name']}': Low und High vertauscht – automatisch korrigiert.")

# Centerpoints: nur möglich wenn mindestens ein Faktor stetig ist
_cp_faktoren = [f for f in _fak_doe if f.get("centerpoint_moeglich", True)]
if not _cp_faktoren:
    if centerpoints > 0:
        print("ℹ️ Keine stetigen Faktoren → Centerpoints nicht möglich, auf 0 gesetzt.")
    centerpoints = 0
elif centerpoints < 0:
    centerpoints = 0

_bin_namen = [f["name"] for f in _fak_doe if not f.get("centerpoint_moeglich", True)]
if _bin_namen:
    print(f"ℹ️ Zweistufige Faktoren (kein Centerpoint): {', '.join(_bin_namen)}")

projekt.versuchsplan_config = {
    "wiederholungen": wiederholungen,
    "blocking": blocking,
    "centerpoints": centerpoints,
    "design": _design,
}

projekt.versuchsplan = helper.generiere_versuchsplan(
    _fak_doe,
    wiederholungen=wiederholungen,
    blocking=blocking,
    centerpoints=centerpoints,
    seed=projekt.seed,
    design=_design,
)

helper.zeige_versuchsplan_info(projekt.versuchsplan, _fak_doe)
print(f"\nDie ersten 10 Versuche:")
display(projekt.versuchsplan.head(10))
helper.speichere_fortschritt(projekt)


<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Was bedeutet die Kodierung (−1 / +1)?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

In der Versuchsmatrix werden die Faktorstufen **kodiert**:
- **−1** = Low-Stufe (euer unterer Wert)
- **+1** = High-Stufe (euer oberer Wert)
- **0** = Centerpoint (Mittelwert zwischen Low und High)

Die Kodierung hat einen wichtigen Vorteil: Alle Faktoren sind auf die gleiche Skala normiert. Dadurch kann man die **Effektgrößen direkt vergleichen** – egal ob ein Faktor in Grad und der andere in Zentimetern gemessen wird.

Ein Koeffizient von +25 bei kodierter Eingabe bedeutet: Wenn der Faktor von −1 auf +1 wechselt (also von Low auf High), steigt die Wurfweite um 25 cm.
</div>
</details>

In [ ]:
#@title 📥 Versuchsplan als Excel herunterladen
filepath = helper.erstelle_doe_excel(
    projekt.versuchsplan, helper._effektive_faktoren(projekt)
)
print(f"✅ Excel erstellt!")

# Kopie im Drive-Ordner ablegen
import shutil
_save_dir = helper._fortschritt_verzeichnis(projekt)
if _save_dir:
    os.makedirs(_save_dir, exist_ok=True)
    shutil.copy2(filepath, os.path.join(_save_dir, os.path.basename(filepath)))

try:
    from google.colab import files
    files.download(filepath)
    print("📥 Download gestartet!")
except ImportError:
    print(f"📁 Datei: {filepath}")


---
## ⏱️ Versuchsdurchführung

Jetzt geht es los! Führt die Versuche in der **randomisierten Reihenfolge** aus dem Excel durch.

**Pro Versuch (~2 Min):**
1. Faktoren laut Plan einstellen
2. Wurf durchführen
3. Auftreffpunkt markieren
4. Wurfweite messen
5. Ergebnis ins Excel eintragen
6. → Nächster Versuch

> ⚠️ **Wichtig:** Reihenfolge einhalten! Die Randomisierung schützt vor systematischen Verfälschungen.

> 📋 **Für den Bericht:** Dokumentiert besondere Vorkommnisse (Katapult-Probleme, Ausreißer-Würfe, etc.)

---
# Phase 3: ANALYZE – Auswertung

## Was sagen die Daten?

Bevor ihr die Ergebnisse seht, hier die drei wichtigsten Werkzeuge:

### Pareto-Diagramm (Hauptwerkzeug)
Das Pareto-Diagramm sortiert alle Effekte nach Stärke. **Je länger der Balken, desto größer der Einfluss** des Faktors auf die Wurfweite. Balken über der roten Linie sind **statistisch signifikant**.

### p-Wert (Signifikanzfilter)
Der p-Wert beantwortet die Frage: *Könnte dieser Effekt auch Zufall sein?*
- **p < 0,05** → Der Effekt ist real (statistisch signifikant)
- **p > 0,05** → Nicht sicher genug – könnte Zufall sein

### R² (Modellgüte)
R² sagt euch: **Wie viel Prozent der Streuung erklärt euer Modell?**
- R² = 0,80 bedeutet: 80% der Variation wird durch eure Faktoren erklärt
- Die restlichen 20% sind Rauschen (Messungenauigkeit, unbekannte Einflüsse)

Ladet jetzt eure ausgefüllte Versuchsergebnis-Excel hoch.

In [ ]:
#@title ⬆️ Versuchsergebnisse hochladen
try:
    from google.colab import files
    print("⬆️ Bitte DoE_Versuchsergebnisse.xlsx (ausgefüllt) hochladen:")
    uploaded = files.upload()
    doe_file = list(uploaded.keys())[0]
except ImportError:
    doe_file = "DoE_Versuchsergebnisse.xlsx"

doe_df = pd.read_excel(doe_file)
print(f"✅ {len(doe_df)} Versuchsergebnisse geladen")
display(doe_df.head())

# In Projekt speichern
projekt.doe_ergebnisse = doe_df
projekt.csv_daten["doe_ergebnisse"] = doe_df

# Excel-Backup auf Drive speichern
import shutil
_save_dir = helper._fortschritt_verzeichnis(projekt)
if _save_dir:
    os.makedirs(_save_dir, exist_ok=True)
    shutil.copy2(doe_file, os.path.join(_save_dir, "DoE_Versuchsergebnisse.xlsx"))

helper.speichere_fortschritt(projekt)


In [ ]:
#@title 📊 Regressionsmodell berechnen
try:
    # Faktoren aus Excel erkennen, falls weder Master- noch DoE-Liste vorhanden
    _fak = helper._effektive_faktoren(projekt)
    if not _fak and projekt.doe_ergebnisse is not None:
        _fak = helper._parse_faktoren_aus_excel(projekt.doe_ergebnisse)
        projekt.faktoren_doe = _fak
        if _fak:
            print(f"ℹ️ {len(_fak)} Faktoren aus Excel erkannt:")
            for f in _fak:
                print(f"   • {f['name']} [{f['low']} – {f['high']} {f['einheit']}]")

    projekt.modell = helper.fitte_modell(projekt.doe_ergebnisse, _fak)
    print(f"✅ Modell berechnet!")
    print(f"   R² = {projekt.modell.rsquared:.4f}")
    print(f"   Adj. R² = {projekt.modell.rsquared_adj:.4f}")
    print(f"   Faktoren: {', '.join(projekt.modell._faktor_namen)}")
    helper.speichere_fortschritt(projekt)
except Exception as e:
    print(f"❌ Fehler bei der Modellberechnung: {e}")
    print("\nPrüft eure Daten:")
    print("  - Enthält die Excel-Datei eine Spalte 'Ergebnis: Weite (cm)'?")
    print("  - Sind alle Werte Zahlen (keine leeren Zellen, kein Text)?")
    print("  - Habt ihr Dezimalpunkte statt Kommas verwendet?")
    if projekt.doe_ergebnisse is not None:
        print(f"\n  Spalten in der Excel: {projekt.doe_ergebnisse.columns.tolist()}")
    _fak = helper._effektive_faktoren(projekt)
    if _fak:
        print(f"  Verwendete Faktoren: {[f['name'] for f in _fak]}")
    else:
        print("  ⚠️ Keine Faktoren definiert! Bitte DEFINE → 'Faktoren übernehmen' ausführen.")


In [ ]:
#@title ⚙️ Automatisches Modell-Pruning (Hierarchieprinzip)
projekt.modell_gepruned, projekt.pruning_log = helper.hierarchisches_pruning(projekt.modell)

print("Pruning-Protokoll:")
for msg in projekt.pruning_log:
    print(f"  {msg}")

# Modell aktualisieren
if projekt.modell_gepruned is not None:
    projekt.modell = projekt.modell_gepruned
    print(f"\n✅ Finales Modell: R² = {projekt.modell.rsquared:.4f}")

<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Was ist das Hierarchieprinzip beim Pruning?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">
Ein Interaktionseffekt bedeutet, dass die Wirkung von Faktor A davon abhängt, auf welcher Stufe Faktor B steht. Ohne den Haupteffekt A im Modell wäre die Interaktion nicht korrekt interpretierbar. Deshalb bleibt Faktor A im Modell, auch wenn er allein nicht signifikant ist.

**Beispiel:** Wenn die Interaktion "Winkel × Spannung" signifikant ist (p < 0.05), bleiben sowohl "Winkel" als auch "Spannung" im Modell – auch wenn einer der beiden Haupteffekte allein p > 0.05 hat.
</div>
</details>

In [ ]:
#@title 📊 Pareto-Diagramm der standardisierten Effekte
fig = helper.plot_pareto_effekte(projekt.modell)
helper._save_fig(projekt, fig, "analyze_pareto")
plt.show()

In [ ]:
#@title 📊 Koeffiziententabelle
helper.zeige_koeffizienten(projekt.modell)

# Warnung wenn kein Faktor signifikant
p_vals = projekt.modell.pvalues.drop("Intercept", errors="ignore")
if (p_vals > 0.05).all():
    display(HTML("""
    <div style="padding:10px; border-left:4px solid #DC2626; background:#FEE2E2; border-radius:4px; margin:8px 0;">
        ❌ <strong>Kein Faktor ist signifikant (alle p > 0,05).</strong><br>
        Mögliche Ursachen:<br>
        • Stufenabstände zu klein – war der Unterschied zwischen Low und High spürbar?<br>
        • Zu wenig Wiederholungen – Effekte gehen im Rauschen unter<br>
        • Falsche Faktoren gewählt – relevanter Parameter wurde nicht variiert<br>
        <strong>→ Trotzdem weitermachen!</strong> Interpretiert die Richtung der Effekte und diskutiert die Ursachen im Bericht.
    </div>"""))

In [ ]:
#@title 📊 Modellgüte
helper.zeige_modellguete(projekt.modell)

In [ ]:
#@title ⚙️ Hintergrundprüfungen (VIF, Lack-of-Fit)
# VIF
vifs = helper.pruefe_vif(projekt.modell)

# Lack-of-Fit (nur wenn Centerpoints vorhanden)
if projekt.doe_ergebnisse is not None:
    lof = helper.pruefe_lack_of_fit(projekt.modell, projekt.modell._daten)

In [ ]:
#@title 🔍 Residuenplots + ANOVA-Tabelle (Prio 2)
fig = helper.plot_residuen(projekt.modell)
helper._save_fig(projekt, fig, "analyze_residuen")
plt.show()

# ANOVA-Tabelle (Prio 2)
helper.zeige_anova_tabelle(projekt.modell)


<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Wie liest man Residuenplots?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

**Residuen** = Gemessener Wert – Vorhergesagter Wert. Sie zeigen, was das Modell **nicht** erklären kann.

**Gute Zeichen:**
- Residuen zufällig um Null verteilt (keine Muster)
- Keine Trichterform (konstante Varianz)
- Punkte nahe der Linie im Q-Q-Plot (Normalverteilung)

**Schlechte Zeichen:**
- Systematische Muster (z.B. U-Form) → nichtlinearer Effekt fehlt
- Trichterform → Varianz hängt von der Größe ab
- Einzelne Ausreißer weit weg → eventuell Messfehler
</div>
</details>

<div style="padding:10px; border-left:4px solid #2563EB; background:#DBEAFE; border-radius:4px;">
📋 <strong>Für den Bericht:</strong> Pareto-Diagramm, Koeffiziententabelle und R² sind die drei zentralen Outputs der Analyze-Phase. Exportiert sie am Ende des Tages.
</div>

In [ ]:
#@title 💾 Ergebnisse bis ANALYZE auf Google Drive speichern
helper.exportiere_phase_auf_drive(projekt, "ANALYZE")


---
# Phase 4: IMPROVE

## Optimale Einstellung finden und bestätigen

Das Modell sagt euch, welche Faktoren wichtig sind. Jetzt nutzen wir es, um die **optimale Einstellung** für eure Zielweite zu finden.

In [ ]:
#@title 📊 Konturplot (Wurfweite)
faktor_x = 1 #@param {type:"integer"}
faktor_y = 2 #@param {type:"integer"}

# Faktornamen anzeigen zur Orientierung
_fn = projekt.modell._faktor_namen
print("Verfügbare Faktoren:")
for i, name in enumerate(_fn, 1):
    print(f"  {i}: {name}")
print(f"\nGewählt: X-Achse = {_fn[faktor_x - 1]}, Y-Achse = {_fn[faktor_y - 1]}")

if faktor_x == faktor_y:
    print("❌ Bitte zwei verschiedene Faktoren wählen!")
else:
    fig = helper.plot_kontur(
        projekt.modell, projekt.faktoren, projekt.zielweite,
        faktor_idx=(faktor_x - 1, faktor_y - 1)
    )
    helper._save_fig(projekt, fig, f"improve_kontur_{_fn[faktor_x-1]}_{_fn[faktor_y-1]}")
    plt.show()


## 📊 Varianz-Konturplots

Der Konturplot oben zeigt, **wo** ihr eure Zielweite trefft (Mittelwert). Aber wo ist der Prozess auch **konsistent**?

Zwei Ansätze:
- **Dispersionsmodell:** Modelliert die tatsächliche Streuung aus euren Wiederholungen
- **Fehlerfortpflanzung:** Berechnet analytisch, wo Einstellfehler die Wurfweite am wenigsten beeinflussen

In [ ]:
#@title 📊 Varianz-Konturplot: Dispersionsmodell
faktor_x = 1 #@param {type:"integer"}
faktor_y = 2 #@param {type:"integer"}

_fn = projekt.modell._faktor_namen
print(f"Faktoren: {', '.join(f'{i+1}={n}' for i, n in enumerate(_fn))}")
print(f"Gewählt: X={_fn[faktor_x-1]}, Y={_fn[faktor_y-1]}")

if faktor_x != faktor_y:
    _df = projekt.modell._daten
    _doe_X = _df[_fn].values
    _doe_response = _df['Y'].values

    fig = helper.plot_kontur_varianz_dispersion(
        projekt.modell, _doe_X, _doe_response,
        projekt.faktoren,
        faktor_idx=(faktor_x - 1, faktor_y - 1)
    )
    helper._save_fig(projekt, fig, f"improve_varianz_disp_{_fn[faktor_x-1]}_{_fn[faktor_y-1]}")
    plt.show()
else:
    print("❌ Bitte zwei verschiedene Faktoren wählen!")


In [ ]:
#@title 📊 Varianz-Konturplot: Fehlerfortpflanzung
faktor_x = 1 #@param {type:"integer"}
faktor_y = 2 #@param {type:"integer"}

_fn = projekt.modell._faktor_namen
print(f"Faktoren: {', '.join(f'{i+1}={n}' for i, n in enumerate(_fn))}")
print(f"Gewählt: X={_fn[faktor_x-1]}, Y={_fn[faktor_y-1]}")

if faktor_x != faktor_y:
    fig = helper.plot_kontur_varianz_transmitted(
        projekt.modell, projekt.faktoren,
        faktor_idx=(faktor_x - 1, faktor_y - 1)
    )
    helper._save_fig(projekt, fig, f"improve_varianz_trans_{_fn[faktor_x-1]}_{_fn[faktor_y-1]}")
    plt.show()
else:
    print("❌ Bitte zwei verschiedene Faktoren wählen!")


In [ ]:
#@title 🎯 Optimale Einstellungen berechnen (analytisch)
# Strategie wählen: "mittelwert", "varianz", oder "dual"
strategie = "dual" #@param ["mittelwert", "varianz", "dual"] {type:"string"}
lambda_gewicht = 0.01 #@param {type:"number"}

projekt.optimale_einstellung = helper.optimiere_einstellungen(
    projekt.modell, projekt.zielweite, projekt.faktoren,
    strategie=strategie, lambda_gewicht=lambda_gewicht
)
helper.zeige_optimierung(projekt.optimale_einstellung)

projekt.optimierung_config = {
    "strategie": strategie,
    "lambda_gewicht": lambda_gewicht,
}
helper.speichere_fortschritt(projekt)


In [ ]:
#@title 📐 Regressionsformel anzeigen
helper.zeige_regressionsformel(projekt.modell, projekt.faktoren)

### 🔮 Prognosetool

Nutzt das Regressionsmodell, um für **beliebige Faktoreinstellungen** die Wurfweite vorherzusagen.

- Tragt eure gewünschten Werte als **kodierte Werte** ein (−1 = niedrig, +1 = hoch, 0 = Mitte)
- Oder als **Originalwerte** (z.B. Winkel in Grad) — die Kodierung erfolgt automatisch
- Die Kodierungstabelle findet ihr in der Formel-Anzeige oben

In [ ]:
#@title 🔮 Prognosetool: Wurfweite vorhersagen
import ipywidgets as _widgets

# Slider pro Faktor erzeugen
_faktor_sliders = []
_slider_box = []
for _f in projekt.faktoren:
    _center = (_f["low"] + _f["high"]) / 2
    _half = (_f["high"] - _f["low"]) / 2
    _slider = _widgets.FloatSlider(
        value=_center,
        min=_f["low"] - 0.2 * _half,
        max=_f["high"] + 0.2 * _half,
        step=round(_half / 10, 2) or 0.1,
        description=f'{_f["name"]} ({_f["einheit"]})',
        style={'description_width': '180px'},
        layout=_widgets.Layout(width='500px'),
        readout_format='.1f',
    )
    # Kodierter Wert als Label daneben
    _code_label = _widgets.Label(value="(kodiert: 0.00)")
    def _update_label(change, lbl=_code_label, f=_f):
        c = (f["low"] + f["high"]) / 2
        h = (f["high"] - f["low"]) / 2
        coded = (change["new"] - c) / h if h > 0 else 0
        lbl.value = f"(kodiert: {coded:+.2f})"
    _slider.observe(_update_label, names='value')
    _faktor_sliders.append(_slider)
    _slider_box.append(_widgets.HBox([_slider, _code_label]))

_output = _widgets.Output()

def _berechne_prognose(_btn):
    _output.clear_output()
    with _output:
        werte = {}
        for _f, _s in zip(projekt.faktoren, _faktor_sliders):
            werte[_f["name"]] = _s.value
        ergebnis = helper.prognostiziere(
            projekt.modell, projekt.faktoren, werte
        )
        helper.zeige_prognose(
            ergebnis, projekt.faktoren, werte,
            zielweite=projekt.zielweite
        )

_btn = _widgets.Button(
    description='🔮 Prognose berechnen',
    button_style='primary',
    layout=_widgets.Layout(width='250px', height='38px', margin='12px 0')
)
_btn.on_click(_berechne_prognose)

# Anzeige
display(HTML(f"<p><strong>Zielweite:</strong> {projekt.zielweite:.0f} cm ± {projekt.toleranz:.0f} cm</p>"))
for _box in _slider_box:
    display(_box)
display(_btn)
display(_output)

# Initial-Prognose auslösen
_berechne_prognose(None)

### 🔍 Visuelle vs. analytische Optimierung

- **Visuell (Konturplot):** Ihr seht die Antwortfläche und könnt das Optimum abschätzen. Gut für Verständnis, aber ungenau.
- **Analytisch (Optimizer):** Der Computer findet die exakte Einstellung. Genauer, aber als Black Box weniger lehrreich.

➡️ Der Konturplot hilft euch zu **verstehen**, der Optimizer liefert die **exakten Werte**.

In [ ]:
#@title 📊 Vergleich: Alle drei Optimierungsstrategien
ergebnisse = helper.vergleiche_optimierungen(
    projekt.modell, projekt.zielweite, projekt.faktoren
)
# Das Ergebnis der gewählten Strategie wird für die Konfirmation verwendet.
print(f"\n→ Für die Konfirmation wird die Strategie '{strategie}' verwendet.")

> 💡 **Robustheitshinweis:** Sucht Einstellungen, bei denen nicht nur der Mittelwert stimmt, sondern auch die **Streuung zwischen euren Wiederholungen klein war**. Prüft die Standardabweichungen in eurer Versuchstabelle – eine Kombination mit kleiner Streuung ist robuster als eine mit hohem Mittelwert aber großer Schwankung.

<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Konfidenzintervall vs. Vorhersageintervall
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

Das Notebook zeigt ein **Vorhersageintervall** (prediction interval), nicht ein Konfidenzintervall:

- **Konfidenzintervall:** Wo liegt der wahre Mittelwert? (schmaler)
- **Vorhersageintervall:** Wo wird der nächste einzelne Wurf landen? (breiter)

Das Vorhersageintervall ist breiter, weil es nicht nur die Unsicherheit über den Mittelwert, sondern auch die **natürliche Streuung** des Prozesses berücksichtigt.

Für die Konfirmation ist das Vorhersageintervall das richtige Maß: Ihr wollt wissen, ob eure **einzelnen Würfe** in den erwarteten Bereich fallen.
</div>
</details>

In [ ]:
#@title 📥 Konfirmations-Template herunterladen
filepath = helper.erstelle_konfirmation_excel(
    projekt.optimale_einstellung["einstellungen"],
    projekt.zielweite,
)
print(f"✅ Template erstellt!")

# Kopie im Drive-Ordner ablegen
import shutil
_save_dir = helper._fortschritt_verzeichnis(projekt)
if _save_dir:
    os.makedirs(_save_dir, exist_ok=True)
    shutil.copy2(filepath, os.path.join(_save_dir, os.path.basename(filepath)))

try:
    from google.colab import files
    files.download(filepath)
except ImportError:
    print(f"📁 Datei: {filepath}")

print(f"\n🎯 Stellt euer Katapult auf die empfohlenen Einstellungen und macht mindestens 10 Würfe!")


In [ ]:
#@title 📝 Konfirmationswürfe eingeben (Weite in cm)
wurf_01 = 0.0 #@param {type:"number"}
wurf_02 = 0.0 #@param {type:"number"}
wurf_03 = 0.0 #@param {type:"number"}
wurf_04 = 0.0 #@param {type:"number"}
wurf_05 = 0.0 #@param {type:"number"}
wurf_06 = 0.0 #@param {type:"number"}
wurf_07 = 0.0 #@param {type:"number"}
wurf_08 = 0.0 #@param {type:"number"}
wurf_09 = 0.0 #@param {type:"number"}
wurf_10 = 0.0 #@param {type:"number"}
wurf_11 = 0.0 #@param {type:"number"}
wurf_12 = 0.0 #@param {type:"number"}
wurf_13 = 0.0 #@param {type:"number"}
wurf_14 = 0.0 #@param {type:"number"}
wurf_15 = 0.0 #@param {type:"number"}
wurf_16 = 0.0 #@param {type:"number"}
wurf_17 = 0.0 #@param {type:"number"}
wurf_18 = 0.0 #@param {type:"number"}
wurf_19 = 0.0 #@param {type:"number"}
wurf_20 = 0.0 #@param {type:"number"}

_alle = [
         wurf_01, wurf_02, wurf_03, wurf_04, wurf_05,
         wurf_06, wurf_07, wurf_08, wurf_09, wurf_10,
         wurf_11, wurf_12, wurf_13, wurf_14, wurf_15,
         wurf_16, wurf_17, wurf_18, wurf_19, wurf_20]
werte = [round(w, 1) for w in _alle if w > 0]

if werte:
    projekt.konfirmation_wuerfe = np.array(werte)
    print(f"✅ {len(werte)} Konfirmationswürfe eingetragen")
    print(f"   Werte: {werte}")
    helper.speichere_fortschritt(projekt)
elif len(projekt.konfirmation_wuerfe) > 0:
    print(f"ℹ️ {len(projekt.konfirmation_wuerfe)} Konfirmationswürfe aus gespeichertem Fortschritt geladen.")
else:
    print("⚠️ Bitte Konfirmationswürfe eingeben (mindestens 10 Würfe empfohlen).")

In [ ]:
#@title 📊 Konfirmation auswerten
if len(projekt.konfirmation_wuerfe) > 0:
    opt = projekt.optimale_einstellung
    ergebnis = helper.analysiere_konfirmation(
        projekt.konfirmation_wuerfe,
        opt["vorhersage"], opt["pi_low"], opt["pi_high"],
        projekt.zielweite, projekt.toleranz
    )
    projekt.konfirmation_ergebnis = ergebnis
    helper.zeige_konfirmation(ergebnis)

    # Zielscheibe
    fig = helper.plot_zielscheibe(
        projekt.konfirmation_wuerfe, projekt.zielweite, projekt.toleranz,
        modus=projekt.messmodus, titel="Konfirmation – Optimierte Einstellung",
        farbe=helper.GREEN
    )
    helper._save_fig(projekt, fig, "improve_konfirmation_zielscheibe")
    plt.show()

    helper.hinweis_bericht("Die Konfirmation ist der zentrale Nachweis, dass der DMAIC-Zyklus funktioniert hat. Zielscheiben-Plot und Bewertung gehören in den Bericht.")

In [ ]:
#@title 💾 Ergebnisse bis IMPROVE auf Google Drive speichern
helper.exportiere_phase_auf_drive(projekt, "IMPROVE")


---
# Phase 5: CONTROL

## Ist der verbesserte Prozess stabil und fähig?

In der Control-Phase prüft ihr:
1. **Stabilität:** Läuft der Prozess gleichmäßig? (I-MR-Kontrollkarte)
2. **Normalverteilung:** Voraussetzung für Cpk (Shapiro-Wilk + Q-Q-Plot)
3. **Prozessfähigkeit:** Passt der Prozess in die Spezifikation? (Cpk)
4. **Vorher/Nachher:** Visueller Verbesserungsvergleich

In [ ]:
#@title 📊 I-MR-Kontrollkarte (Stabilitätsprüfung)
if len(projekt.konfirmation_wuerfe) > 0:
    projekt.imr_ergebnis = helper.berechne_imr(projekt.konfirmation_wuerfe)

    fig = helper.plot_imr(projekt.konfirmation_wuerfe)
    helper._save_fig(projekt, fig, "control_imr")
    plt.show()

    helper.zeige_stabilitaet(projekt.imr_ergebnis)

<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Wie werden die Kontrollgrenzen berechnet?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

Die I-MR-Kontrollkarte berechnet die Grenzen aus den **Moving Ranges** (gleitende Spannweiten):

$$MR_i = |x_i - x_{i-1}|$$
$$\overline{MR} = \frac{1}{n-1}\sum MR_i$$

**I-Chart (Einzelwerte):**
$$UCL = \bar{x} + 2{,}66 \cdot \overline{MR}$$
$$LCL = \bar{x} - 2{,}66 \cdot \overline{MR}$$

**MR-Chart:**
$$UCL_{MR} = 3{,}267 \cdot \overline{MR}$$

Der Faktor 2,66 kommt aus d₂ = 1,128 für n=2 (Moving-Range-Subgruppe): 3/d₂ ≈ 2,66.
</div>
</details>

In [ ]:
#@title 📊 Normalverteilungsprüfung (Voraussetzung für Cpk)
if len(projekt.konfirmation_wuerfe) > 0:
    norm_test = helper.pruefe_normalverteilung(projekt.konfirmation_wuerfe)

    fig = helper.plot_qq(projekt.konfirmation_wuerfe)
    helper._save_fig(projekt, fig, "control_qq")
    plt.show()

    if not np.isnan(norm_test['shapiro_p']):
        shapiro_schwellen = [
            (0.05, "⚠️", "Cpk mit Vorsicht interpretieren – Daten möglicherweise nicht normalverteilt"),
            (float('inf'), "✅", "Normalverteilungsannahme beibehalten – Cpk ist aussagekräftig"),
        ]
        helper.zeige_ampel(norm_test['shapiro_p'], shapiro_schwellen,
                          titel="Shapiro-Wilk p-Wert:")

### Prozessfähigkeit (Cpk) – Was bedeutet das?

Der **Cpk** misst, ob euer Prozess dauerhaft in die Spezifikation passt:

$$C_{pk} = \min\left(\frac{USL - \bar{x}}{3\sigma},\; \frac{\bar{x} - LSL}{3\sigma}\right)$$

- **USL** (Upper Specification Limit) = Zielweite + Toleranz
- **LSL** (Lower Specification Limit) = Zielweite − Toleranz

| Cpk | Industrie | Euer Katapult |
|-----|-----------|---------------|
| < 0,67 | ❌ Nicht fähig | ❌ Verbesserungsbedürftig |
| 0,67–1,0 | ❌ Nicht fähig | ⚠️ Verbesserung ggü. Baseline |
| 1,0–1,33 | ⚠️ Bedingt fähig | ✅ Gut für ein Experiment |
| ≥ 1,33 | ✅ Prozess fähig | ✅ Hervorragend |

> *In der Industrie wird Cpk ≥ 1,33 gefordert. Für euer selbstgebautes Katapult ist ein Cpk > 0,67 bereits ein Erfolg.*

In [ ]:
#@title 📊 Prozessfähigkeit (Cpk)
if len(projekt.konfirmation_wuerfe) > 0:
    usl = projekt.zielweite + projekt.toleranz
    lsl = projekt.zielweite - projekt.toleranz

    cpk = helper.berechne_cpk(projekt.konfirmation_wuerfe, usl, lsl)
    projekt.cpk_ergebnis = cpk
    helper.zeige_cpk(cpk)

    fig = helper.plot_cpk_verteilung(cpk)
    helper._save_fig(projekt, fig, "control_cpk_verteilung")
    plt.show()

In [ ]:
#@title 📊 Vorher / Nachher – Zielscheiben-Vergleich
if len(projekt.baseline_wuerfe) > 0 and len(projekt.konfirmation_wuerfe) > 0:
    fig = helper.plot_vorher_nachher(
        projekt.baseline_wuerfe, projekt.konfirmation_wuerfe,
        projekt.zielweite, projekt.toleranz, projekt.messmodus
    )
    helper._save_fig(projekt, fig, "control_vorher_nachher")
    plt.show()

    helper.hinweis_bericht("Cpk-Wert, Kontrollkarte und Vorher/Nachher-Zielscheibe sind die drei zentralen Control-Outputs.")

<details style="margin:10px 0; padding:8px; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px;">
<summary style="cursor:pointer; font-weight:bold; color:#2563EB;">
🔍 Für Neugierige: Was sagt der Cpk-Wert aus?
</summary>
<div style="margin-top:8px; padding:8px; font-size:0.95em;">

Der **Cpk** (Process Capability Index) misst, wie gut euer Prozess in die Spezifikationsgrenzen passt:

$$Cpk = \min\left(\frac{USL - \bar{x}}{3\sigma}, \frac{\bar{x} - LSL}{3\sigma}\right)$$

**Interpretation:**
- Cpk < 0: Mittelwert außerhalb der Spezifikation
- Cpk 0–1.0: Prozess passt nicht sicher in die Toleranz
- Cpk 1.0–1.33: Akzeptabel
- Cpk > 1.33: Gut (industrieller Standard)
- Cpk > 1.67: Exzellent

Ein Cpk von 1.0 bedeutet: 99,73% der Werte liegen innerhalb der Spezifikation (bei Normalverteilung). Bei Cpk = 1.33 sind es 99,994%.
</div>
</details>

In [ ]:
#@title 💾 Ergebnisse bis CONTROL auf Google Drive speichern
helper.exportiere_phase_auf_drive(projekt, "CONTROL")


---
# Abschluss: Lessons Learned

Nehmt euch 5 Minuten und reflektiert euren DMAIC-Prozess.

## 📝 Reflexion: Lessons Learned

Nehmt euch 10 Minuten Zeit, um die folgenden Fragen als Gruppe zu beantworten.
Die Antworten werden in eurem Projektexport gespeichert.


In [ ]:
#@title 📝 Reflexion ausfüllen
import ipywidgets as _widgets

_reflexion_fragen = [
    "Was hat in eurem DMAIC-Projekt gut funktioniert?",
    "Welche Fehlerquellen habt ihr identifiziert (Messung, Durchführung, Modell)?",
    "Was würdet ihr beim nächsten Mal anders machen?",
    "Welche Six-Sigma / QM-Konzepte waren am lehrreichsten und warum?",
    "Offene Fragen oder Verständnisprobleme?",
]

_antwort_widgets = []
for _i, _frage in enumerate(_reflexion_fragen, 1):
    display(HTML(f"<h4 style='color:#2563EB; margin:18px 0 4px 0;'>{_i}. {_frage}</h4>"))
    _w = _widgets.Textarea(
        value='',
        placeholder='Hier eure Antwort eingeben...',
        layout=_widgets.Layout(width='100%', height='120px')
    )
    display(_w)
    _antwort_widgets.append(_w)

def _speichere_reflexion(_btn):
    antworten = [w.value.strip() for w in _antwort_widgets]
    if not any(antworten):
        print("ℹ️ Bitte mindestens eine Frage beantworten.")
        return
    projekt.charter["Lessons Learned"] = "\n\n".join(
        f"{i}. {frage}\n   {antwort}"
        for i, (frage, antwort) in enumerate(zip(_reflexion_fragen, antworten), 1)
    )
    helper.speichere_fortschritt(projekt)
    print("✅ Reflexion gespeichert!")

_btn = _widgets.Button(
    description='💾 Reflexion speichern',
    button_style='primary',
    layout=_widgets.Layout(width='250px', height='38px', margin='20px 0 0 0')
)
_btn.on_click(_speichere_reflexion)
display(_btn)

In [ ]:
#@title 📦 Alle Ergebnisse exportieren (ZIP)
filepath = helper.exportiere_zip(projekt)
print(f"✅ ZIP erstellt: {filepath}")

# Kopie im Drive-Ordner ablegen
import shutil
_save_dir = helper._fortschritt_verzeichnis(projekt)
if _save_dir:
    os.makedirs(_save_dir, exist_ok=True)
    shutil.copy2(filepath, os.path.join(_save_dir, os.path.basename(filepath)))
    print(f"💾 ZIP auch in Google Drive gespeichert.")

try:
    from google.colab import files
    files.download(filepath)
    print("📥 Download gestartet!")
except ImportError:
    print(f"📁 Datei: {filepath}")

print("""
📋 Die ZIP-Datei enthält:
   📂 plots/     – Alle Grafiken als PNG
   📂 daten/     – Alle Datentabellen als CSV
   📄 zusammenfassung.txt – Ergebnisübersicht
""")


---
# 🏁 Geschafft!

Ihr habt heute den **kompletten DMAIC-Zyklus** durchlaufen:

| Phase | ✓ |
|-------|----|
| **Define** | Problem quantifiziert, Charter erstellt |
| **Measure** | Messsystem geprüft, Baseline erhoben |
| **Analyze** | Einflussfaktoren identifiziert, Modell bewertet |
| **Improve** | Optimale Einstellungen gefunden und bestätigt |
| **Control** | Prozessfähigkeit bewertet |

## Nächste Schritte
- **Bericht einreichen bis 30. April 2026**
- Nutzt die exportierten PNGs und CSVs als Grundlage
- Nicht nur Ergebnisse zeigen – **erklären, was sie bedeuten!**

> *Das Ziel dieser Session ist nicht, ein perfektes Katapult zu bauen, sondern den DMAIC-Prozess als Werkzeug für systematische Verbesserung zu erlernen und anzuwenden.*